# Run the OCR comparison in Colab

This runs **EasyOCR and public TrOCR** on the same 271 validation crops and frozen line boundaries. It saves raw text, timing and errors. It does not train models or calculate accuracy. Twenty unresolved layouts remain explicit errors. Paddle is not implemented here; the supplied private fine-tuned TrOCR is a separate experiment.

1. On your Mac, in this repository, run `git bundle create /tmp/ocr-colab.bundle HEAD`. This packages the committed version of your current branch, including its history. Uncommitted changes are excluded.
2. Upload this notebook to Colab. Select **Runtime → Change runtime type → GPU** before running cells.
3. Run the cells in order and upload `ocr-colab.bundle` when prompted. No GitHub token is needed.
4. Download the results before disconnecting. Colab hardware and package compatibility have not been verified by running this notebook yet.


In [ ]:
from google.colab import files
from pathlib import Path
import subprocess, sys, os
uploaded = files.upload()
assert len(uploaded) == 1 and 'ocr-colab.bundle' in uploaded, 'Upload ocr-colab.bundle only'
repo = Path('/content/ocr-project')
assert not repo.exists(), 'Repository already exists; use a fresh runtime'
os.environ['GIT_LFS_SKIP_SMUDGE'] = '1'
subprocess.run(['git', 'clone', '/content/ocr-colab.bundle', str(repo)], check=True)
os.chdir(repo)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)
del uploaded


In [ ]:
# Keep Colab's installed GPU-enabled PyTorch build.
subprocess.run([sys.executable, '-m', 'pip', 'install',
    'easyocr==1.7.2', 'transformers>=4.52,<5', 'safetensors>=0.4',
    'Pillow>=11.3', 'opencv-python-headless>=4.10,<5', 'sentencepiece'], check=True)


In [ ]:
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['nvidia-smi'], check=True)
sys.path.insert(0, str(repo))
from backend.app.evaluation.ocr_crop_baseline import load_crops
from backend.app.evaluation.ocr_line_layout import load_layout
workspace = repo / 'evaluation/ocr_annotations/v4-reviewed-01'
layout = repo / 'evaluation/ocr_layouts/v4-lines-v1/manifest.json'
records = load_crops(workspace)
load_layout(layout, records)
print('Verified crops:', len(records))


In [ ]:
# Download weights before timing. This does not train either model.
import easyocr
cache = repo / '.cache/easyocr'
reader = easyocr.Reader(['en'], gpu=False, detector=False, recognizer=True,
    model_storage_directory=str(cache),
    user_network_directory=str(cache / 'user_network'), verbose=False)
del reader
subprocess.run([sys.executable, 'scripts/fetch_public_trocr.py'], check=True)


In [ ]:
# Both models run sequentially on the same GPU. Keep failed-run files too.
from datetime import datetime, timezone
run_root = Path('/content/ocr-results') / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
run_root.mkdir(parents=True, exist_ok=False)
statuses = {}
for engine in ['easyocr', 'trocr']:
    result = subprocess.run([sys.executable, '-u', 'scripts/run_easyocr_crops.py',
        '--engine', engine, '--device', 'cuda', '--warmup', '3', '--seed', '0',
        '--line-layout', str(layout), '--output', str(run_root / engine)])
    statuses[engine] = result.returncode
    print(engine, 'exit code:', result.returncode)
(run_root / 'exit_codes.json').write_text(__import__('json').dumps(statuses, indent=2))
print('Finished. Exit code 0 means completed; inspect errors and coverage in each report.')


In [ ]:
# Download everything, including errors and provenance, before ending the session.
import shutil
archive = shutil.make_archive(str(run_root), 'zip', root_dir=run_root)
files.download(archive)


## What to hand back

Keep the downloaded ZIP. Each completed model folder contains `predictions.csv`, `report.json`, `environment.json`, input hashes and previews. These are outputs and timing, not accuracy scores. Resolve transcription/group reviews before scoring. Compare latency between these two Colab runs, not against the previous Mac run. Shared AI-reviewed line boundaries are diagnostic assistance, not automatic deployment performance. This notebook does not validate the newer LCD/sensor/A/B/C detector changes.
